# RealMLP + HGBC-TE blend

Tests blending v0.8's RealMLP (neural net, solo OOF 0.9506, the highest raw OOF
in this project) against **v0.7's HGBC-TE** (HistGradientBoosting + exact-value
target encoding, our actual current best model, OOF 0.9502) -- not against
CatBoost-V1 as v0.8's own notebook did. Flagged as the natural follow-up in
`docs/investigate/notebook-runs.md`'s v0.8 entry: RealMLP found a genuine
non-degenerate blend weight against CatBoost-V1 (82-92%, not collapsing to one
model like every earlier blend attempt), suggesting real diversity -- worth
testing whether that diversity clears our threshold when paired with the
actual best model instead of the second-best one.

Both models retrained fresh here (RealMLP from v0.8, HGBC-TE from v0.7, same
configs) using a single shared `StratifiedKFold(5)` so their OOF arrays are
directly comparable and blendable. No CatBoost-V1 reproduction needed this
time -- it's not a blend member here, which keeps the whole run fast on
Kaggle's GPU (RealMLP ~10 min, HGBC-TE ~5 min).

Per explicit request, this submits its result **unconditionally** (regardless
of whether it clears the project's usual 0.0005 real-improvement threshold) --
this run is itself a curiosity/exploration pass, not gated like v0.1-v0.8.

In [ ]:
import os
import glob
import math
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import TargetEncoder, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import HistGradientBoostingClassifier

DATA_DIR = Path('/kaggle/input/playground-series-s6e7')
if not (DATA_DIR / 'train.csv').exists():
    hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if hits:
        DATA_DIR = Path(hits[0]).parent
    else:
        DATA_DIR = Path('..') / 'data'  # local run
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else DATA_DIR
print('DATA_DIR:', DATA_DIR)

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'device: {device}')

def seed_everything(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(2026)

N_FOLDS = 5
EPOCHS = 5
TRAIN_BS = 512
EVAL_BS = 10240
EMBED_DIM = 4
LR = 0.01
N_ENS = 16
ONEHOTMAX = 10
LS = 0.05
EMA_DECAY = 0.997

TARGET = 'health_condition'
NUMERIC_COLS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
                 'step_count', 'exercise_duration', 'water_intake']
CATEGORICAL_COLS = ['diet_type', 'stress_level', 'sleep_quality',
                     'physical_activity_level', 'smoking_alcohol', 'gender']
BASE_FEATURES = NUMERIC_COLS + CATEGORICAL_COLS
RAW_COLS = NUMERIC_COLS + CATEGORICAL_COLS

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print('train', train.shape, 'test', test.shape)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train[TARGET])
n_classes = len(label_encoder.classes_)
print('classes:', list(label_encoder.classes_))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
folds = list(skf.split(train, y))

## Feature engineering -- shared base, two independent views

In [ ]:
# RealMLP view: binned _cat/_cat2 numeric copies + integer-mapped categoricals
train_realmlp = train.copy()
test_realmlp = test.copy()

def FE(df):
    for c in NUMERIC_COLS:
        if c == 'step_count':
            df[f'{c}_cat'] = (df[c] // 10).astype(str)
        elif c == 'calorie_expenditure':
            df[f'{c}_cat'] = (df[c] // 5).astype(str)
        else:
            df[f'{c}_cat'] = df[c].astype(str)
    for c in NUMERIC_COLS:
        if c == 'step_count':
            df[f'{c}_cat2'] = (df[c] // 20).astype(str)
        elif c == 'calorie_expenditure':
            df[f'{c}_cat2'] = (df[c] // 50).astype(str)
        elif c == 'water_intake':
            df[f'{c}_cat2'] = (df[c] * 50).astype(np.int64).astype(str)
        elif c in ('heart_rate', 'bmi'):
            df[f'{c}_cat2'] = (df[c] * 5).astype(np.int64).astype(str)
        else:
            df[f'{c}_cat2'] = (df[c] // 2).astype(str)
    return df

for c in CATEGORICAL_COLS:
    train_realmlp[c] = train_realmlp[c].fillna('missing').astype(str)
    test_realmlp[c] = test_realmlp[c].fillna('missing').astype(str)
for c in NUMERIC_COLS:
    train_realmlp[c] = train_realmlp[c].fillna(0)
    test_realmlp[c] = test_realmlp[c].fillna(0)

train_realmlp = FE(train_realmlp)
test_realmlp = FE(test_realmlp)

CATS = [c for c in test_realmlp.columns if not pd.api.types.is_numeric_dtype(test_realmlp[c]) or train_realmlp[c].nunique() <= ONEHOTMAX]
NUMS = [c for c in test_realmlp.columns if c not in CATS and c != TARGET and c != 'id']
print(f'RealMLP: {len(CATS)} categorical (incl. binned), {len(NUMS)} numeric')

CAT_DIMS = []
for c in CATS:
    mapping = {v: i + 1 for i, v in enumerate(train_realmlp[c].unique())}
    train_realmlp[c] = train_realmlp[c].map(mapping)
    test_realmlp[c] = test_realmlp[c].map(mapping).fillna(0).astype(np.int64)
    CAT_DIMS.append(int(train_realmlp[c].max()) + 1)

# HGBC-TE view: native pandas.Categorical for model input, all 13 raw cols as
# exact-value strings for target encoding (matches v0.7 exactly)
train_model = train.copy()
test_model = test.copy()
for col in CATEGORICAL_COLS:
    categories = sorted(x for x in (set(train_model[col].unique()) | set(test_model[col].unique())) if pd.notna(x))
    train_model[col] = pd.Categorical(train_model[col], categories=categories)
    test_model[col] = pd.Categorical(test_model[col], categories=categories)

train_te_str = train[RAW_COLS].astype(str).fillna('na')
test_te_str = test[RAW_COLS].astype(str).fillna('na')

## RealMLP architecture

Unchanged from v0.8 -- periodic numeric embeddings, NTK-parametrized linears,
16-way ensemble-in-one-model, EMA.

In [ ]:
class RobustScaleSmoothClipTransform(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self._median = np.median(X, axis=-2)
        quant_diff = np.quantile(X, 0.75, axis=-2) - np.quantile(X, 0.25, axis=-2)
        idxs = quant_diff == 0.0
        quant_diff[idxs] = 0.5 * (np.max(X, axis=-2)[idxs] - np.min(X, axis=-2)[idxs])
        factors = 1.0 / (quant_diff + 1e-30)
        factors[quant_diff == 0.0] = 0.0
        self._factors = factors
        return self

    def transform(self, X, y=None):
        x_scaled = self._factors[None, :] * (X - self._median[None, :])
        return x_scaled / np.sqrt(1 + (x_scaled / 3) ** 2)


class ScalingLayer(nn.Module):
    def __init__(self, n_ens, n_features):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(n_ens, n_features))

    def forward(self, x):
        return x * self.scale[None, :, :]


class CategoricalFeatureLayer(nn.Module):
    def __init__(self, n_ens, cat_dims, embed_dim=8, device=None):
        super().__init__()
        self.n_ens = n_ens
        self.embed_dim = embed_dim
        self.cat_dims = cat_dims

        self.onehot_features, self.binary_features = [], []
        self.embed_features, self.embed_dims, self.embed_offsets = [], [], []

        for i, dim in enumerate(cat_dims):
            if dim == 2:
                self.binary_features.append(i)
            elif dim <= ONEHOTMAX:
                self.onehot_features.append(i)
            else:
                self.embed_features.append(i)
                self.embed_dims.append(dim)

        self.combined_emb = None
        if self.embed_features:
            total_vocab = int(sum(self.embed_dims) * n_ens)
            self.combined_emb = nn.Embedding(total_vocab, self.embed_dim, padding_idx=0)
            offset = 0
            for dim in self.embed_dims:
                self.embed_offsets.append(offset)
                offset += dim
            self.per_ens_offset = sum(self.embed_dims)

    def forward(self, x):
        batch_size, n_ens, n_cat = x.shape
        features = []

        if self.binary_features:
            binary_x = x[:, :, self.binary_features].float()
            features.append(2 * binary_x - 1)

        if self.onehot_features:
            onehot_x = x[:, :, self.onehot_features]
            onehot_dims = [self.cat_dims[i] for i in self.onehot_features]
            total_onehot_dim = sum(onehot_dims)
            onehot_encoded = torch.zeros(batch_size, n_ens, total_onehot_dim, device=x.device)
            start = 0
            for idx, dim in enumerate(onehot_dims):
                pos = onehot_x[:, :, idx:idx + 1].long()
                pos = torch.clamp(pos, 0, dim - 1)
                onehot_encoded.scatter_(2, pos + start, 1.0)
                start += dim
            features.append(onehot_encoded)

        if self.embed_features and self.combined_emb is not None:
            embed_x = x[:, :, self.embed_features].long()
            ens_offset = torch.arange(n_ens, device=x.device) * self.per_ens_offset
            feat_offset = torch.tensor(self.embed_offsets, device=x.device)
            indices = embed_x + feat_offset.unsqueeze(0).unsqueeze(0) + ens_offset.unsqueeze(0).unsqueeze(-1)
            embedded = self.combined_emb(indices)
            features.append(embedded.view(batch_size, n_ens, -1))

        return torch.cat(features, dim=2)


class PBLDEmbedding(nn.Module):
    def __init__(self, n_ens, n_features, hidden_dim=16, out_dim=4, freq_scale=0.1):
        super().__init__()
        self.n_ens, self.n_features, self.out_dim = n_ens, n_features, out_dim
        self.w1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim) * freq_scale)
        self.b1 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim))
        self.w2 = nn.Parameter(torch.randn(n_ens, n_features, hidden_dim, out_dim - 1) * (1.0 / np.sqrt(hidden_dim)))
        self.b2 = nn.Parameter(torch.randn(n_ens, n_features, out_dim - 1))
        self.act = nn.GELU()
        nn.init.uniform_(self.b1, -np.pi, np.pi)

    def forward(self, x):
        batch_size = x.shape[0]
        x_expanded = x.unsqueeze(-1)
        periodic = torch.cos(2 * np.pi * (x_expanded * self.w1.unsqueeze(0) + self.b1.unsqueeze(0)))
        transformed = torch.einsum('b n f h, n f h d -> b n f d', periodic, self.w2)
        transformed = self.act(transformed + self.b2.unsqueeze(0))
        result = torch.cat([x.unsqueeze(-1), transformed], dim=-1)
        return result.view(batch_size, self.n_ens, -1)


class NTPLinear(nn.Module):
    def __init__(self, n_ens, in_features, out_features, bias=True):
        super().__init__()
        self.in_features, self.out_features = in_features, out_features
        self.weight = nn.Parameter(torch.randn(n_ens, in_features, out_features))
        self.bias = nn.Parameter(torch.randn(n_ens, out_features)) if bias else None

    def forward(self, x):
        x = torch.einsum('b n i, n i o -> b n o', x, self.weight) / np.sqrt(self.in_features)
        if self.bias is not None:
            x = x + self.bias
        return x


class RankGLUHead(nn.Module):
    def __init__(self, input_dim, bottleneck_dim=128, output_dim=1, gamma=0.5, n_ens=8):
        super().__init__()
        self.gamma = gamma
        self.layer_norm = nn.LayerNorm(input_dim)
        self.linear_score = nn.Linear(input_dim, output_dim)
        self.glu_v = nn.Linear(input_dim, bottleneck_dim)
        self.glu_g = nn.Linear(input_dim, bottleneck_dim)
        self.glu_out = nn.Linear(bottleneck_dim, output_dim)

    def forward(self, x):
        normalized = self.layer_norm(x)
        linear_part = self.linear_score(normalized)
        gate = torch.sigmoid(self.glu_g(normalized))
        value = self.glu_v(normalized)
        glu_output = self.glu_out(value * gate)
        return linear_part + self.gamma * glu_output


class RealMLP(nn.Module):
    def __init__(self, output_dim=3, cat_dims=(), n_numerical=None, n_ens=8, embed_dim=4):
        super().__init__()
        act = nn.GELU
        self.n_ens, self.embed_dim, self.output_dim = n_ens, embed_dim, output_dim

        self.cate = CategoricalFeatureLayer(n_ens=n_ens, cat_dims=cat_dims, embed_dim=embed_dim)
        self.num_embed = PBLDEmbedding(n_features=n_numerical, hidden_dim=20, out_dim=5,
                                        freq_scale=5.0, n_ens=n_ens)
        num_emb_dim = n_numerical * 5
        cat_emb_dim = sum(c if c <= ONEHOTMAX else embed_dim for c in cat_dims)
        total_dim = num_emb_dim + cat_emb_dim

        self.dropout = nn.Dropout(0.03)
        self.first_linear = NTPLinear(n_ens=n_ens, in_features=total_dim, out_features=256)
        self.model = nn.Sequential(
            ScalingLayer(n_ens=n_ens, n_features=total_dim), self.dropout,
            self.first_linear,
            NTPLinear(n_ens=n_ens, in_features=256, out_features=512), act(),
            NTPLinear(n_ens=n_ens, in_features=512, out_features=128), act(), self.dropout,
        )
        self.cls_head = RankGLUHead(input_dim=128, bottleneck_dim=128, output_dim=output_dim,
                                     gamma=0.5, n_ens=n_ens)

    def forward(self, x_num, x_cat):
        x_num = x_num.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_cat = x_cat.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_num = self.num_embed(x_num)
        x_cat = self.cate(x_cat)
        x = self.model(torch.cat([x_num, x_cat], dim=2))
        return self.cls_head(x)


def get_parameter_groups_5class(model):
    first_linear_weight_id = id(model.first_linear.weight)
    scale_p, pbld_p, first_w_p, other_w_p, bias_p = [], [], [], [], []
    for name, param in model.named_parameters():
        if 'scale' in name:
            scale_p.append(param)
        elif 'num_embed' in name:
            pbld_p.append(param)
        elif id(param) == first_linear_weight_id:
            first_w_p.append(param)
        elif 'bias' in name:
            bias_p.append(param)
        else:
            other_w_p.append(param)
    return scale_p, pbld_p, first_w_p, other_w_p, bias_p


def flat_anneal(init_value, progress, flat_ratio=0.5):
    if progress < flat_ratio:
        return init_value
    decay_progress = (progress - flat_ratio) / (1 - flat_ratio)
    return init_value * (1 - decay_progress)


def cos_schedule(init_value, progress):
    return init_value * (math.cos(math.pi * progress) + 1) / 2

## RealMLP training

In [ ]:
X_cat_all = train_realmlp[CATS].values
oof_proba_realmlp = np.zeros((len(train_realmlp), n_classes))
test_proba_realmlp = np.zeros((len(test_realmlp), n_classes))

fold_bar = tqdm(folds, desc='RealMLP folds')
for fold, (tr_idx, val_idx) in enumerate(fold_bar):

    X_num_tr, X_cat_tr, y_tr = train_realmlp[NUMS].values[tr_idx], X_cat_all[tr_idx], y[tr_idx]
    X_num_val, X_cat_val, y_val = train_realmlp[NUMS].values[val_idx], X_cat_all[val_idx], y[val_idx]
    X_num_test, X_cat_test = test_realmlp[NUMS].values.copy(), test_realmlp[CATS].values.copy()

    te_cols = [c for c in CATS if 'cat' in c]
    encoder = TargetEncoder(cv=5, smooth='auto', target_type='multiclass')
    tr_enc = encoder.fit_transform(pd.DataFrame(X_cat_tr, columns=CATS)[te_cols], y_tr)
    val_enc = encoder.transform(pd.DataFrame(X_cat_val, columns=CATS)[te_cols])
    tst_enc = encoder.transform(pd.DataFrame(X_cat_test, columns=CATS)[te_cols])

    X_num_tr = np.concatenate([X_num_tr, tr_enc], axis=1)
    X_num_val = np.concatenate([X_num_val, val_enc], axis=1)
    X_num_test = np.concatenate([X_num_test, tst_enc], axis=1)

    unique_classes = np.unique(y_tr)
    fold_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_tr)
    fold_class_weights = torch.tensor(fold_class_weights * np.array([0.9, 1.1, 1.0]),
                                       dtype=torch.float32).to(device)
    fold_bar.set_postfix(class_weights=[round(w, 3) for w in fold_class_weights.tolist()])

    rssc = RobustScaleSmoothClipTransform()
    rssc.fit(X_num_tr)
    X_num_tr = rssc.transform(X_num_tr)
    X_num_val = rssc.transform(X_num_val)
    X_num_test = rssc.transform(X_num_test)

    realmlp = RealMLP(output_dim=n_classes, cat_dims=[d for d in CAT_DIMS],
                       n_numerical=X_num_tr.shape[1], n_ens=N_ENS, embed_dim=EMBED_DIM).to(device)

    scale_p, pbld_p, first_w_p, other_w_p, bias_p = get_parameter_groups_5class(realmlp)
    optimizer = torch.optim.AdamW([
        {'params': scale_p,   'lr': LR * 20.0,  'weight_decay': 1e-2 * 0.1},
        {'params': pbld_p,    'lr': LR * 0.093, 'weight_decay': 1e-2},
        {'params': first_w_p, 'lr': LR * 1.0,   'weight_decay': 1e-2 * 0.1},
        {'params': other_w_p, 'lr': LR,         'weight_decay': 1e-2},
        {'params': bias_p,    'lr': LR * 0.1,   'weight_decay': 1e-2 * 0.5},
    ], betas=(0.9, 0.98))

    X_num_tr_t = torch.as_tensor(X_num_tr, dtype=torch.float32, device=device)
    X_cat_tr_t = torch.as_tensor(X_cat_tr, dtype=torch.long, device=device)
    y_tr_t = torch.as_tensor(y_tr, dtype=torch.long, device=device)
    X_num_val_t = torch.as_tensor(X_num_val, dtype=torch.float32, device=device)
    X_cat_val_t = torch.as_tensor(X_cat_val, dtype=torch.long, device=device)
    X_num_test_t = torch.as_tensor(X_num_test, dtype=torch.float32, device=device)
    X_cat_test_t = torch.as_tensor(X_cat_test, dtype=torch.long, device=device)

    train_indices = np.arange(len(y_tr_t))
    ema_state = {k: v.detach().clone() for k, v in realmlp.state_dict().items()}

    best_acc, best_model_state = 0.0, None

    epoch_bar = tqdm(range(EPOCHS), desc=f'fold {fold} epochs', leave=False)
    for epoch in epoch_bar:
        batch_starts = range(0, len(y_tr_t), TRAIN_BS)
        epoch_loss_total, epoch_loss_n = 0.0, 0
        for idx in batch_starts:
            progress = epoch / EPOCHS + idx / (len(y_tr_t) * EPOCHS)
            bs_idx = train_indices[idx:idx + TRAIN_BS]
            realmlp.train()

            optimizer.param_groups[0]['lr'] = flat_anneal(LR * 20.0, progress)
            optimizer.param_groups[1]['lr'] = flat_anneal(LR * 0.093, progress)
            optimizer.param_groups[2]['lr'] = flat_anneal(LR * 1.0, progress)
            optimizer.param_groups[3]['lr'] = flat_anneal(LR, progress)
            optimizer.param_groups[4]['lr'] = flat_anneal(LR * 0.1, progress)

            optimizer.zero_grad()
            batch_x_num, batch_x_cat, batch_y = X_num_tr_t[bs_idx], X_cat_tr_t[bs_idx], y_tr_t[bs_idx]
            logits = realmlp(batch_x_num, batch_x_cat)
            loss = F.cross_entropy(
                logits.reshape(-1, n_classes), batch_y.repeat_interleave(N_ENS),
                weight=fold_class_weights, label_smoothing=cos_schedule(LS, progress),
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(realmlp.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss_total += loss.item()
            epoch_loss_n += 1

            with torch.no_grad():
                model_state = realmlp.state_dict()
                for key, value in model_state.items():
                    if torch.is_floating_point(value):
                        ema_state[key].mul_(EMA_DECAY).add_(value.detach(), alpha=1.0 - EMA_DECAY)
                    else:
                        ema_state[key].copy_(value)

        if epoch:
            realmlp.eval()
            live_state = {k: v.detach().clone() for k, v in realmlp.state_dict().items()}
            realmlp.load_state_dict(ema_state, strict=True)
            with torch.no_grad():
                all_probs = []
                for vi in range(0, len(y_val), EVAL_BS):
                    logits = realmlp(X_num_val_t[vi:vi + EVAL_BS], X_cat_val_t[vi:vi + EVAL_BS])
                    all_probs.append(F.softmax(logits, dim=-1).mean(dim=1).cpu().numpy())
                all_probs = np.concatenate(all_probs, axis=0)
                bal_acc = balanced_accuracy_score(y_val, all_probs.argmax(axis=1))
                if bal_acc > best_acc:
                    best_acc = bal_acc
                    best_model_state = {k: v.cpu().clone() for k, v in ema_state.items()}
            realmlp.load_state_dict(live_state, strict=True)

        epoch_bar.set_postfix(loss=f'{epoch_loss_total / max(epoch_loss_n, 1):.4f}', best_acc=f'{best_acc:.5f}')
        np.random.shuffle(train_indices)

    if best_model_state is not None:
        realmlp.load_state_dict(best_model_state)
    realmlp = realmlp.to(device)
    realmlp.eval()
    with torch.no_grad():
        all_probs = []
        for vi in range(0, len(y_val), EVAL_BS):
            logits = realmlp(X_num_val_t[vi:vi + EVAL_BS], X_cat_val_t[vi:vi + EVAL_BS])
            all_probs.append(F.softmax(logits, dim=-1).mean(dim=1).cpu().numpy())
        oof_proba_realmlp[val_idx] = np.concatenate(all_probs, axis=0)

        all_probs = []
        for ti in range(0, len(X_num_test_t), EVAL_BS):
            logits = realmlp(X_num_test_t[ti:ti + EVAL_BS], X_cat_test_t[ti:ti + EVAL_BS])
            all_probs.append(F.softmax(logits, dim=-1).mean(dim=1).cpu().numpy())
        test_proba_realmlp += np.concatenate(all_probs, axis=0) / N_FOLDS

oof_pred_realmlp = oof_proba_realmlp.argmax(axis=1)
oof_bal_acc_realmlp = balanced_accuracy_score(y, oof_pred_realmlp)
print(f'\nRealMLP OOF balanced accuracy: {oof_bal_acc_realmlp:.5f}')

## HGBC-TE training (v0.7's exact config)

In [ ]:
HGBC_CONFIG = dict(
    learning_rate=0.0627037115235577,
    max_iter=300,
    max_leaf_nodes=33,
    min_samples_leaf=298,
    l2_regularization=0.028912644384523085,
    max_bins=237,
    max_features=0.820265066682815,
    early_stopping=True,
    categorical_features='from_dtype',
    class_weight='balanced',
    random_state=42,
)

te_names = [f'te_{c}_{k}' for c in RAW_COLS for k in range(n_classes)]
oof_proba_hgbc = np.zeros((len(train_model), n_classes))
test_proba_hgbc = np.zeros((len(test_model), n_classes))

for fold, (tr_idx, val_idx) in enumerate(tqdm(folds, desc='HGBC-TE folds')):
    enc = TargetEncoder(cv=5, smooth='auto', target_type='multiclass')
    te_tr = enc.fit_transform(train_te_str.iloc[tr_idx], y[tr_idx])
    te_val = enc.transform(train_te_str.iloc[val_idx])
    te_test = enc.transform(test_te_str)

    X_tr = pd.concat([train_model[BASE_FEATURES].iloc[tr_idx].reset_index(drop=True),
                       pd.DataFrame(te_tr, columns=te_names)], axis=1)
    X_val = pd.concat([train_model[BASE_FEATURES].iloc[val_idx].reset_index(drop=True),
                        pd.DataFrame(te_val, columns=te_names)], axis=1)
    X_test = pd.concat([test_model[BASE_FEATURES].reset_index(drop=True),
                         pd.DataFrame(te_test, columns=te_names)], axis=1)
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = HistGradientBoostingClassifier(**HGBC_CONFIG)
    model.fit(X_tr, y_tr)
    oof_proba_hgbc[val_idx] = model.predict_proba(X_val)
    val_pred = oof_proba_hgbc[val_idx].argmax(axis=1)
    fold_score = balanced_accuracy_score(y_val, val_pred)
    print(f'fold {fold}: balanced accuracy = {fold_score:.5f} (n_iter={model.n_iter_})')
    test_proba_hgbc += model.predict_proba(X_test) / N_FOLDS

oof_pred_hgbc = oof_proba_hgbc.argmax(axis=1)
oof_bal_acc_hgbc = balanced_accuracy_score(y, oof_pred_hgbc)
print(f'\nHGBC-TE OOF balanced accuracy: {oof_bal_acc_hgbc:.5f}')
print('v0.7 known result: 0.9502')

## Solo comparison

In [ ]:
solo_scores = {'realmlp': oof_bal_acc_realmlp, 'hgbc_te': oof_bal_acc_hgbc}
for name, score in sorted(solo_scores.items(), key=lambda kv: -kv[1]):
    print(f'{score:.4f}  {name}')

## Blend weight search + nested validation

In [ ]:
OOF_PROBAS = {'realmlp': oof_proba_realmlp, 'hgbc_te': oof_proba_hgbc}
TEST_PROBAS = {'realmlp': test_proba_realmlp, 'hgbc_te': test_proba_hgbc}
MEMBER_NAMES = ['realmlp', 'hgbc_te']

def pair_grid(step=0.02):
    n_steps = round(1 / step)
    return [(i * step, (n_steps - i) * step) for i in range(n_steps + 1)]

def blend_predict(probas_dict, weights, members):
    blended = sum(w * probas_dict[m] for w, m in zip(weights, members))
    return blended.argmax(axis=1)

def grid_search_blend(oof_probas_dict, y_true, members, grid):
    best_score, best_w = -1, tuple(1 / len(members) for _ in members)
    for w in tqdm(grid, desc=f'blend grid ({"+".join(members)})', leave=False):
        pred = blend_predict(oof_probas_dict, w, members)
        score = balanced_accuracy_score(y_true, pred)
        if score > best_score:
            best_score, best_w = score, w
    return best_score, best_w

grid2 = pair_grid(step=0.02)
best_score_2way, best_w_2way = grid_search_blend(OOF_PROBAS, y, MEMBER_NAMES, grid2)
print(f'2-way blend full-OOF grid search: best {best_score_2way:.4f} at weights {dict(zip(MEMBER_NAMES, (round(x,2) for x in best_w_2way)))}')
print(f'Best solo model:                  {max(solo_scores.values()):.4f}')
print(f'Improvement (same-data, likely optimistic): {best_score_2way - max(solo_scores.values()):+.4f}')

In [ ]:
nested_scores_solo_best = []
nested_scores_blend = []
nested_weights_2way = []
best_solo_key = max(solo_scores, key=solo_scores.get)

for fold_idx, (_, val_idx) in enumerate(tqdm(folds, desc='nested CV (2-way blend)')):
    other_idx = np.concatenate([folds[j][1] for j in range(N_FOLDS) if j != fold_idx])
    other_oof = {m: OOF_PROBAS[m][other_idx] for m in MEMBER_NAMES}
    _, w_fold = grid_search_blend(other_oof, y[other_idx], MEMBER_NAMES, grid2)

    val_oof = {m: OOF_PROBAS[m][val_idx] for m in MEMBER_NAMES}
    blend_pred = blend_predict(val_oof, w_fold, MEMBER_NAMES)
    solo_pred = OOF_PROBAS[best_solo_key][val_idx].argmax(axis=1)

    nested_scores_solo_best.append(balanced_accuracy_score(y[val_idx], solo_pred))
    nested_scores_blend.append(balanced_accuracy_score(y[val_idx], blend_pred))
    nested_weights_2way.append(w_fold)

print('Per-fold blend weights (fit on the other 4 folds):', [tuple(round(x, 2) for x in w) for w in nested_weights_2way])
print(f'Nested best-solo-model ({best_solo_key}) balanced accuracy: {np.mean(nested_scores_solo_best):.4f} (+/- {np.std(nested_scores_solo_best):.4f})')
print(f'Nested 2-way-blend balanced accuracy:                      {np.mean(nested_scores_blend):.4f} (+/- {np.std(nested_scores_blend):.4f})')
print(f'Honest improvement estimate: {np.mean(nested_scores_blend) - np.mean(nested_scores_solo_best):+.4f}')

## Decision + candidate submission

Per explicit request, this writes a submission **unconditionally** using
whichever of (blend, best solo) scored higher in nested validation -- this
run is an exploration pass, not gated behind the usual 0.0005 threshold.

In [ ]:
honest_improvement = np.mean(nested_scores_blend) - np.mean(nested_scores_solo_best)
CURRENT_BEST_OOF = 0.9502  # v0.7 HGBC-TE, our current best model

print(f'RealMLP solo:  {oof_bal_acc_realmlp:.4f}')
print(f'HGBC-TE solo:  {oof_bal_acc_hgbc:.4f}')
print(f'Current project best OOF: {CURRENT_BEST_OOF:.4f}')
print()

if np.mean(nested_scores_blend) >= np.mean(nested_scores_solo_best):
    print(f'Using 2-way blend (nested: {np.mean(nested_scores_blend):.4f}, honest improvement {honest_improvement:+.4f}). Weights fit on full OOF: {dict(zip(MEMBER_NAMES, best_w_2way))}')
    test_pred = blend_predict(TEST_PROBAS, best_w_2way, MEMBER_NAMES)
else:
    print(f'Using solo {best_solo_key} (nested blend did not beat solo).')
    test_pred = TEST_PROBAS[best_solo_key].argmax(axis=1)

submission = pd.DataFrame({'id': test['id'], TARGET: label_encoder.inverse_transform(test_pred)})
assert list(submission.columns) == list(sample_submission.columns)
assert len(submission) == len(sample_submission)
submission_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)
print(f'wrote {submission_path} ({len(submission)} rows)')
display(submission[TARGET].value_counts(normalize=True))

## Summary

**Highest LB score in this project (0.95065), but not a confirmed new best
model.** Run on Kaggle GPU (T4).

- **RealMLP solo OOF: 0.9506**, **HGBC-TE solo OOF: 0.9503** (matches v0.7
  closely).
- Blend (86% realmlp / 14% hgbc_te): 0.9507 same-data. **Nested-validated
  honest improvement: ~+0.0000** -- essentially no measurable value added
  over RealMLP solo, matching v0.8's RealMLP+CatBoost-V1 result (+0.0001).
- Submitted unconditionally (per request, no threshold gate): **public LB
  0.95065** -- the highest LB in this project so far, ahead of v0.8's
  0.95048 and v0.7's confirmed-best 0.95036.
- **Caveat**: despite the highest LB number, this is not treated as a
  confirmed new best -- the ~0 nested improvement means the blend doesn't
  meaningfully beat RealMLP solo under honest cross-validation. The higher
  LB reflects RealMLP's own strong solo performance, not new diversity from
  this pairing.

**v0.7 (HGBC-TE) and v0.8 (RealMLP) remain statistically tied for best
model.** RealMLP has now failed to find real ensemble lift with two
different tree-boosting partners (CatBoost-V1 in v0.8, HGBC-TE here) --
further blend attempts would need a structurally different partner model to
have a real chance at adding diversity.